# Efficient VLM Inference via Visual Token Compression

Colab demo for Qwen2.5-VL / Qwen2-VL inference-time visual token compression benchmarks.

## 1. Install dependencies

Use an A100 runtime when possible. Colab normally ships with a CUDA-matched PyTorch build, so the torch install line is kept as an optional fallback.

In [ ]:
try:
    import torch
except ImportError:
    %pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124

%pip install -q -U torchvision transformers accelerate qwen-vl-utils datasets pillow pandas matplotlib tqdm psutil pynvml pyyaml

# If you see KeyError: 'qwen2_5_vl' or 'qwen2_vl', run this and restart runtime:
# %pip install -q -U git+https://github.com/huggingface/transformers accelerate

## 2. Clone repo and move to project root

This clones the public GitHub repo into Colab and adds the repo root to `sys.path`, so imports like `from src.compression import create_compression_method` work.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/YuqiWang26/vlm.git"
PROJECT_DIR = "/content/vlm"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], check=False)

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("Current directory:", os.getcwd())
print("Repo root added to sys.path:", PROJECT_DIR)

## 3. Check GPU

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("Memory GB:", round(props.total_memory / 1024**3, 1))
!nvidia-smi

## 4. Smoke test compression functions without loading the VLM

In [ ]:
import torch
from src.compression import create_compression_method

device = "cuda" if torch.cuda.is_available() else "cpu"
tokens = torch.randn(1, 64, 128, device=device)
for name in ["none", "fixed", "importance", "merging"]:
    method = create_compression_method(name, retention_ratio=0.5, apply_proxy_image_budget=False)
    compressed = method.compress_visual_tokens(tokens).tokens
    print(name, tuple(tokens.shape), "->", tuple(compressed.shape))

## 5. Quick benchmark

This loads the model and runs a tiny benchmark on synthetic images. On A100, start with Qwen2.5-VL-3B. If model download or memory is a problem, use the Qwen2-VL-2B fallback cell.

In [ ]:
!python run_benchmark.py --quick --model-id Qwen/Qwen2.5-VL-3B-Instruct --dtype fp16 --attn-implementation eager --methods none,fixed --ratios 1.0,0.5

In [ ]:
# Fallback option:
# !python run_benchmark.py --quick --model-id Qwen/Qwen2-VL-2B-Instruct --dtype fp16 --attn-implementation eager --methods none,fixed --ratios 1.0,0.5

## 6. Full benchmark

In [ ]:
# This can take a while. Reduce --samples or remove high/4-image settings if needed.
# !python run_benchmark.py \
#   --methods none,fixed,importance,merging \
#   --ratios 1.0,0.75,0.5,0.25,0.1 \
#   --resolutions low,medium,high \
#   --num-images 1,2,4 \
#   --samples 3 \
#   --max-new-tokens 64

## 7. View results and plots

In [ ]:
import os
import pandas as pd
from IPython.display import Image as IPImage, display

raw_path = "results/benchmark_results.csv"
summary_path = "results/summary_results.csv"

if not os.path.exists(raw_path):
    raise FileNotFoundError("Run the quick benchmark cell before viewing results.")

raw_df = pd.read_csv(raw_path)
display(raw_df.head())

if os.path.exists(summary_path):
    display(pd.read_csv(summary_path))

if "success" in raw_df.columns and not raw_df["success"].astype(bool).any():
    print("No successful benchmark rows. First errors:")
    error_cols = [col for col in ["compression_method", "retention_ratio", "oom", "error"] if col in raw_df.columns]
    display(raw_df[error_cols].head(10))

!python plot_results.py
for filename in [
    "latency_vs_retention_ratio.png",
    "memory_vs_retention_ratio.png",
    "quality_vs_retention_ratio.png",
    "efficiency_accuracy_tradeoff.png",
]:
    path = f"results/{filename}"
    if os.path.exists(path):
        display(IPImage(filename=path))
    else:
        print(f"Plot not created: {path}")